# 00 — Setup smoke test

**Phase 0** of the Ancient Chinese Search Engine notebook chain (see `.cursor/plans/ancient-chinese-search-engine_*.plan.md` §1.5).

Goal: confirm that the three external services the platform depends on are reachable, then bring the Neo4j schema online idempotently.

1. **Silra** (`apps.backend.llm.silra`) — chat + embedding probes via the OpenAI-compatible client.
2. **Neo4j** (`apps.backend.graph.neo4j_client`) — server version + edition.
3. **MinIO** (`apps.backend.storage.minio_client`) — list buckets, ensure the default page-image bucket exists.
4. **Schema** (`apps.backend.graph.schema.init_schema`) — create constraints + vector indexes (idempotent).
5. **Persist** the merged health report to `notebooks/_artifacts/00_setup_smoke_test/health.json` so the next notebook (`00a_philological_seeds.ipynb`) can refuse to advance if anything is red.

Per plan §1.5 rules: this notebook **imports** production modules, never copies their logic. To re-run from a fresh clone, start the stack with `docker compose up -d neo4j redis minio` and `uv run jupyter lab` from the repo root.

In [9]:
from __future__ import annotations

import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'could not locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loaded = load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '00_setup_smoke_test'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo root      : {REPO_ROOT}')
print(f'.env loaded    : {loaded}')
print(f'artifact dir   : {ARTIFACT_DIR}')
print(f'EMBEDDING_DIMS : {os.getenv("EMBEDDING_DIMS", "<unset>")}')
print(f'NEO4J_URI      : {os.getenv("NEO4J_URI", "<unset>")}')
print(f'MINIO_ENDPOINT : {os.getenv("MINIO_ENDPOINT", "<unset>")}')

repo root      : /Users/mohasani/Ancient
.env loaded    : True
artifact dir   : /Users/mohasani/Ancient/notebooks/_artifacts/00_setup_smoke_test
EMBEDDING_DIMS : 1024
NEO4J_URI      : bolt://localhost:7687
MINIO_ENDPOINT : localhost:9000


## Probe 1 — Silra

Sends a 1-token chat completion via `CHAT_LLM_MODEL` and a 1-character embedding via `EMBED_LLM_MODEL`. The OCR model is read from env but not exercised (no image input here).

In [10]:
from apps.backend.llm.silra import ping as silra_ping

silra_health = silra_ping()
print(json.dumps(silra_health, ensure_ascii=False, indent=2))

2026-05-18 22:46:57,705 INFO httpx: HTTP Request: POST https://api.silra.cn/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-18 22:46:57,708 INFO apps.backend.llm.silra: silra.chat model=deepseek-chat prompt=162 completion=1 total=163
2026-05-18 22:46:59,657 INFO httpx: HTTP Request: POST https://api.silra.cn/v1/embeddings "HTTP/1.1 200 OK"
2026-05-18 22:46:59,855 INFO apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=2 total=2


{
  "ok": true,
  "base_url": "https://api.silra.cn/v1/",
  "chat_model": "deepseek-chat",
  "embed_model": "text-embedding-v4",
  "ocr_model": "deepseek-ocr",
  "embedding_dims_expected": 1024,
  "embedding_dims_observed": 1024,
  "chat_sample": "OK",
  "errors": []
}


## Probe 2 — Neo4j

Opens a Bolt session, calls `dbms.components()`, and counts existing constraints + vector indexes (so we can compare before/after schema bring-up).

In [11]:
from apps.backend.graph.neo4j_client import get_driver, ping as neo4j_ping

driver = get_driver()
neo4j_health = neo4j_ping(driver)
print(json.dumps(neo4j_health, ensure_ascii=False, indent=2))

{
  "ok": true,
  "uri": "bolt://localhost:7687",
  "server_version": "5.18.1",
  "edition": "community",
  "database": "neo4j",
  "constraint_count": 16,
  "vector_index_count": 5,
  "errors": []
}


## Probe 3 — MinIO

Lists buckets and ensures `MINIO_BUCKET_PAGES` (default `ancient-pages`) exists. This is the bucket Phase 1 (`01_ingestion.ipynb`) will start writing rasterized PDF pages and EPUB image resources to.

In [12]:
from apps.backend.storage.minio_client import ping as minio_ping

minio_health = minio_ping()
print(json.dumps(minio_health, ensure_ascii=False, indent=2))

{
  "ok": true,
  "endpoint": "localhost:9000",
  "secure": false,
  "buckets": [
    "ancient-pages"
  ],
  "default_bucket": "ancient-pages",
  "default_bucket_created": false,
  "errors": []
}


## Bring schema online (idempotent)

Creates uniqueness constraints on every node label that has a natural key and the five vector indexes from plan §5. Re-running this cell is safe — every statement uses `IF NOT EXISTS`. Embedding dimension is read from `EMBEDDING_DIMS` (the user's `.env` sets `2048`, matching `text-embedding-v4`).

In [13]:
from apps.backend.graph.schema import init_schema

schema_report = init_schema(driver)
print(f'embedding dims: {schema_report["embedding_dims"]}  similarity: {schema_report["similarity"]}')
print(f'constraints   : {len(schema_report["constraints"])}')
for name in schema_report['constraints']:
    print(f'  - {name}')
print(f'vector indexes: {len(schema_report["vector_indexes"])}')
for vi in schema_report['vector_indexes']:
    print(f'  - {vi["name"]:38s} {vi["label"]}.{vi["property"]:24s} dims={vi["dims"]} sim={vi["similarity"]} state={vi["state"]}')
if schema_report['errors']:
    print('\nERRORS:')
    for err in schema_report['errors']:
        print(f'  ! {err}')

2026-05-18 22:46:59,894 INFO neo4j.notifications: Received notification from DBMS server: <GqlStatusObject gql_status='03N42', status_description='`CONSTRAINT user_id_unique FOR (e:User) REQUIRE (e.id) IS UNIQUE` already exists.', position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'SCHEMA', '_severity': 'INFORMATION'}> for query: 'CREATE CONSTRAINT user_id_unique IF NOT EXISTS FOR (n:USER) REQUIRE n.id IS UNIQUE'
2026-05-18 22:46:59,897 INFO neo4j.notifications: Received notification from DBMS server: <GqlStatusObject gql_status='03N42', status_description='`CONSTRAINT topic_id_unique FOR (e:Topic) REQUIRE (e.id) IS UNIQUE` already exists.', position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, 

embedding dims: 1024  similarity: cosine
constraints   : 16
  - chapter_id_unique
  - chunk_id_unique
  - community_id_unique
  - correction_id_unique
  - dictionary_entry_id_unique
  - document_id_unique
  - few_shot_example_id_unique
  - keyword_name_unique
  - metrics_snapshot_ts_unique
  - norm_id_unique
  - page_id_unique
  - problem_class_code_unique
  - section_id_unique
  - topic_id_unique
  - user_id_unique
  - verifier_failure_id_unique
vector indexes: 5
  - chunk_embedding_classical              Chunk.embeddingClassical       dims=1024 sim=COSINE state=ONLINE
  - chunk_embedding_vernacular             Chunk.embeddingVernacular      dims=1024 sim=COSINE state=ONLINE
  - community_summary_embedding_index      Community.embedding                dims=1024 sim=COSINE state=ONLINE
  - keyword_embedding_index                Keyword.embedding                dims=1024 sim=COSINE state=ONLINE
  - page_image_embedding_index             Page.imageEmbedding           dims=1024 sim=COSINE

## Persist health artifact for the next notebook

In [14]:
health = {
    'stage': '00_setup_smoke_test',
    'ts': datetime.now(timezone.utc).isoformat(),
    'silra': silra_health,
    'neo4j': neo4j_health,
    'minio': minio_health,
    'schema': schema_report,
    'env': {
        'EMBEDDING_DIMS': os.getenv('EMBEDDING_DIMS'),
        'CHAT_LLM_MODEL': os.getenv('CHAT_LLM_MODEL'),
        'EMBED_LLM_MODEL': os.getenv('EMBED_LLM_MODEL'),
        'OCR_LLM_MODEL': os.getenv('OCR_LLM_MODEL'),
        'NEO4J_URI': os.getenv('NEO4J_URI'),
        'MINIO_ENDPOINT': os.getenv('MINIO_ENDPOINT'),
    },
}
out = ARTIFACT_DIR / 'health.json'
out.write_text(json.dumps(health, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'wrote {out} ({out.stat().st_size} bytes)')

wrote /Users/mohasani/Ancient/notebooks/_artifacts/00_setup_smoke_test/health.json (5904 bytes)


## Assertions

Hard-fail if any probe is red so the notebook chain refuses to advance with a half-broken environment. Subsequent notebooks (`00a` onward) should `json.load` the artifact above and check `silra.ok && neo4j.ok && minio.ok` before doing real work.

In [15]:
import os
print(os.getenv("MINIO_ENDPOINT"))   # should be localhost:9000
print(os.getenv("MINIO_ROOT_USER"))  # should be AncientChina

localhost:9000
AncientChina


In [16]:
assert silra_health['ok'], f'Silra probe failed: {silra_health["errors"]}'
assert neo4j_health['ok'], f'Neo4j probe failed: {neo4j_health["errors"]}'
assert minio_health['ok'], f'MinIO probe failed: {minio_health["errors"]}'
assert not schema_report['errors'], f'Schema bring-up failed: {schema_report["errors"]}'
assert len(schema_report['constraints']) >= 15, 'expected >= 15 constraints; check schema.CONSTRAINTS'
assert len(schema_report['vector_indexes']) >= 5, 'expected >= 5 vector indexes; check schema.VECTOR_INDEXES'
print('all probes ok — Phase 0 environment is healthy. Next: 00a_philological_seeds.ipynb')

driver.close()

all probes ok — Phase 0 environment is healthy. Next: 00a_philological_seeds.ipynb
